# A/B-тестирование нового интерфейса интернет-магазина

**Цель:** оценить результаты A/B-теста и влияние нового интерфейса сайта интернет-магазина на конверсию пользователей в покупателей, а также определить, привело ли упрощение интерфейса к статистически значимому увеличению конверсии.

**Задачи:**
* Проверить корректность проведения A/B-теста и качество экспериментальных данных.
* Определить период анализа и подготовить данные для оценки результатов эксперимента.
* Оценить изменение конверсии пользователей в покупателей между контрольной и тестовой группами.
* Провести статистическую проверку различий в конверсии.
* Определить, оказало ли изменение интерфейса статистически значимое влияние на ключевую бизнес-метрику.
* Сформулировать итоговый вывод и принять решение о целесообразности внедрения нового интерфейса.

## Описание данных

Для исследования используются два датасета:

* Данные об участниках A/B-тестов, содержащие информацию о распределении пользователей между экспериментальными группами, названии теста и используемом устройстве;
* Данные о действиях пользователей, содержащие информацию о времени и типах событий, совершённых пользователями в течение исследуемого периода.

## Содержимое проекта

* Загрузка данных и знакомство с ними
* Оцените корректность проведения теста
* Исследование данных о пользовательской активности
* Оценка результатов A/B-теста
* Выводы по результатам A/B-тестирования

# Загрузка данных и знакомство с ними

In [1]:
import pandas as pd
import scipy.stats as st
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.proportion import proportions_ztest

In [2]:
ab_test_participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
ab_test_events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip', parse_dates=['event_dt'], low_memory=False)

In [3]:
# Делаем датасет с тестом interface_eu_test
interface_test = ab_test_participants[ab_test_participants['ab_test'] == 'interface_eu_test']
interface_test.head()

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
2,001064FEAAB631A1,A,interface_eu_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac
5,002412F1EB3F6E38,B,interface_eu_test,Mac
6,002540BE89C930FB,B,interface_eu_test,Android


# Оцените корректность проведения теста

In [4]:
# Распределение по группам А и В
groups = (interface_test.groupby('group')['user_id'].nunique().reset_index(name='count'))
groups['share %'] = (groups['count'] / groups['count'].sum()*100).round(2)
groups

,group,count,share %
0,A,5383,49.61
1,B,5467,50.39


In [5]:
# Проверяем пересечения в тесте interface_test
group_a = interface_test.loc[interface_test['group'] == 'A']
group_b = interface_test.loc[interface_test['group'] == 'B']

users_a = set(group_a['user_id'])
users_b = set(group_b['user_id'])

user_overlap = len(users_a & users_b)

print(f'Количество пересекающихся пользователей: {user_overlap}')
print(f'Доля от группы А: {user_overlap / len(group_a):.2%}')
print(f'Доля от группы B: {user_overlap / len(group_b):.2%}')

Количество пересекающихся пользователей: 0
Доля от группы А: 0.00%
Доля от группы B: 0.00%


**Вывод:** Пересечений пользователей между контрольной группой A и тестовой группой B не обнаружено. Это означает, что каждый пользователь участвует только в одной группе теста, что соответствует требованиям корректного проведения A/B-теста.

In [6]:
# Проверяем пересечение среди тестов
tests = ab_test_participants['ab_test'].unique()
print(f'Проводимые тесты: {tests}')
print('---'*30)

interface_users = set(ab_test_participants.loc[ab_test_participants['ab_test']== 'interface_eu_test']['user_id'])
recommender_users = set(ab_test_participants.loc[ab_test_participants['ab_test']== 'recommender_system_test']['user_id'])

overlap_users = interface_users & recommender_users

print(f'Количество пересекающихся пользователей в тестах: {len(overlap_users)}')
print(f'Доля от группы interface_users: {len(overlap_users) / len(interface_users):.2%}')
print(f'Доля от группы recommender_users: {len(overlap_users) / len(recommender_users):.2%}')

Проводимые тесты: ['interface_eu_test' 'recommender_system_test']
------------------------------------------------------------------------------------------
Количество пересекающихся пользователей в тестах: 887
Доля от группы interface_users: 8.18%
Доля от группы recommender_users: 24.14%


**Вывод:** В данных одновременно проводилось два теста: interface_eu_test и recommender_system_test. Было обнаружено 887 пользователей, участвующих в обоих тестах одновременно. Они составляют 8,18% участников interface_eu_test и 24,14% участников recommender_system_test. Наличие пересечений может повлиять на чистоту результатов экспериментов, так как на поведение пользователей одновременно могут воздействовать изменения из двух разных тестов.

In [7]:
# Убираем пересечения в interface_users
interface_test_clean = interface_test[~interface_test['user_id'].isin(overlap_users)]

display(interface_test_clean.head())

print('---'*20)
print(f'Количество строк было до исключения пересечиний: {interface_test.shape[0]}')
print(f'Количество строк стало после исключения пересечиний: {interface_test_clean.shape[0]}')
print(f'Разница: {interface_test.shape[0] - interface_test_clean.shape[0]}')
print('---'*20)
print(f'Пересечение пользователей в тестах после исключения пересечений: {len(set(interface_test_clean["user_id"]) & recommender_users)}')
print('---'*20)

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
4,001E72F50D1C48FA,A,interface_eu_test,Mac
5,002412F1EB3F6E38,B,interface_eu_test,Mac
6,002540BE89C930FB,B,interface_eu_test,Android
7,0031F1B5E9FBF708,A,interface_eu_test,Android


------------------------------------------------------------
Количество строк было до исключения пересечиний: 10850
Количество строк стало после исключения пересечиний: 9963
Разница: 887
------------------------------------------------------------
Пересечение пользователей в тестах после исключения пересечений: 0
------------------------------------------------------------


In [8]:
#  Перераспределение в interface_users после исключения пересечений
print('Распределение в тесте interface_eu_test после исключения пересечений')
groups_clean = (interface_test_clean.groupby('group')['user_id'].nunique().reset_index(name='count'))
groups_clean['share %'] = (groups_clean['count'] / groups_clean['count'].sum()*100).round(2)
groups_clean

Распределение в тесте interface_eu_test после исключения пересечений


,group,count,share %
0,A,4952,49.7
1,B,5011,50.3


**Вывод:** После исключения 887 пользователей, одновременно участвовавших в двух тестах, в исследуемом тесте осталось 9 963 пользователя. Повторная проверка показала отсутствие пересечений с конкурирующим тестом. Распределение пользователей между группами осталось равномерным: в группе A - 4 952 пользователя (49,7%), в группе B - 5 011 пользователей (50,3%). Таким образом, исключение пересекающихся пользователей не привело к существенному дисбалансу между группами.

# Исследование данных о пользовательской активности

In [9]:
display(ab_test_events.head())
print('---'*27)
print(f'Количество строк: {ab_test_events.shape[0]}')

,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


---------------------------------------------------------------------------------
Количество строк: 787286


## Фильтрация данных по пользователям в изучаемом тетсте

In [10]:
# Фильтруем данные по пользователям в изучаемом тетсте
ab_test_events_filter = ab_test_events[ab_test_events['user_id'].isin(interface_test_clean['user_id'])].copy()
display(ab_test_events_filter.head())
print('---'*20)
print(f'Количество строк: {ab_test_events_filter.shape[0]}')

,user_id,event_dt,event_name,details
64672,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0
64946,51278A006E918D97,2020-12-06 14:37:25,registration,-3.8
66585,A0C1E8EFAD874D8B,2020-12-06 17:20:22,registration,-3.32
67873,275A8D6254ACF530,2020-12-06 19:36:54,registration,-0.48
67930,0B704EB2DC7FCA4B,2020-12-06 19:42:20,registration,0.0


------------------------------------------------------------
Количество строк: 73815


In [11]:
# Проверяем фильтрацию по совпадению пользователей в датасетах interface_test_clean и ab_test_events_filter
print(f"Совпадение пользователей: {len(set(interface_test_clean['user_id']) & set(ab_test_events_filter['user_id'])) / interface_test_clean.shape[0] * 100:.2f} %")

Совпадение пользователей: 100.00 %


## Определение периода анализа пользовательской активности

* Рассчитываем время совершения событий относительно даты регистрации.
* Фильтруем события, произошедшие в течение первых семи дней после регистрации пользователя.

In [12]:
# Определяем какие есть события
ab_test_events_filter['event_name'].unique()

array(['registration', 'login', 'product_page', 'purchase',
       'product_cart'], dtype=object)

In [13]:
# Добавляем столбец с датой регестрации каждого пользователя
registration_date = ab_test_events_filter.loc[ab_test_events_filter['event_name']== 'registration'][['user_id','event_dt']]
registration_date = registration_date.rename(columns={'event_dt': 'registration_dt'})

#Соеденяем таблицу ab_test_events_filter и registration_date
ab_test_events_filter = ab_test_events_filter.merge(registration_date, on = 'user_id', how = 'left')

In [14]:
# Считаем дни с момента регистрации
ab_test_events_filter['lifetime'] = (ab_test_events_filter['event_dt']- ab_test_events_filter['registration_dt'])
ab_test_events_filter = ab_test_events_filter.loc[ab_test_events_filter['lifetime'] <= pd.Timedelta(days=7)]
ab_test_events_filter

,user_id,event_dt,event_name,details,registration_dt,lifetime
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0,2020-12-06 14:10:01,0 days 00:00:00
1,51278A006E918D97,2020-12-06 14:37:25,registration,-3.8,2020-12-06 14:37:25,0 days 00:00:00
2,A0C1E8EFAD874D8B,2020-12-06 17:20:22,registration,-3.32,2020-12-06 17:20:22,0 days 00:00:00
3,275A8D6254ACF530,2020-12-06 19:36:54,registration,-0.48,2020-12-06 19:36:54,0 days 00:00:00
4,0B704EB2DC7FCA4B,2020-12-06 19:42:20,registration,0.0,2020-12-06 19:42:20,0 days 00:00:00
...,...,...,...,...,...,...
73667,E89AF4EFC757D283,2020-12-29 21:46:43,product_cart,NaN,2020-12-23 09:35:48,6 days 12:10:55
73670,E89AF4EFC757D283,2020-12-29 21:47:56,product_cart,NaN,2020-12-23 09:35:48,6 days 12:12:08
73739,A6AFDC94A0D3B23D,2020-12-29 22:47:00,product_page,NaN,2020-12-23 13:53:33,6 days 08:53:27
73745,A6AFDC94A0D3B23D,2020-12-29 22:48:46,product_page,NaN,2020-12-23 13:53:33,6 days 08:55:13


## Расчёт необходимого размера выборки

Для оценки достаточности данных и возможности получения статистически значимых результатов рассчитывается необходимый размер выборки для проведения A/B-теста.

Параметры расчёта:

* базовый уровень конверсии - 30%
* статистическая мощность - 80%
* уровень статистической значимости - 5%

In [15]:
alpha = 0.05
power = 0.80
p = 0.3
mde = 0.03

effect_size = proportion_effectsize(p, p + mde)

power_analysis = NormalIndPower()

sample_size = power_analysis.solve_power(
    effect_size = effect_size,
    power = power,
    alpha = alpha,
    ratio = 1)

print(f'Необходимый размер выборки для каждой группы: {int(sample_size)}')

Необходимый размер выборки для каждой группы: 3761


## Расчёт распределения пользователей и покупок по группам

Для оценки различий между контрольной и тестовой группами определим общее количество пользователей и число пользователей, совершивших покупку, в каждой группе.

In [16]:
# Cоеденяем 2 таблицы interface_test_clean и ab_test_events_filter
ab_test = interface_test_clean.merge(ab_test_events_filter, on='user_id', how='left')
ab_test.head()

,user_id,group,ab_test,device,event_dt,event_name,details,registration_dt,lifetime
0,0002CE61FF2C4011,B,interface_eu_test,Mac,2020-12-07 04:37:31,registration,-2.38,2020-12-07 04:37:31,0 days 00:00:00
1,0002CE61FF2C4011,B,interface_eu_test,Mac,2020-12-07 04:37:49,login,NaN,2020-12-07 04:37:31,0 days 00:00:18
2,0002CE61FF2C4011,B,interface_eu_test,Mac,2020-12-07 04:37:57,login,NaN,2020-12-07 04:37:31,0 days 00:00:26
3,0002CE61FF2C4011,B,interface_eu_test,Mac,2020-12-07 04:38:54,login,NaN,2020-12-07 04:37:31,0 days 00:01:23
4,0002CE61FF2C4011,B,interface_eu_test,Mac,2020-12-08 22:15:35,login,NaN,2020-12-07 04:37:31,1 days 17:38:04


In [17]:
# Считаем общее количество в каждой группе и тех кто сделал покупки
total_users = ab_test.groupby('group')['user_id'].nunique()
purchase_users = (ab_test[ab_test['event_name'] == 'purchase'].groupby('group')['user_id'].nunique())

#Делаем датафрейм из двух таблиц
result_test = pd.DataFrame({'total_users': total_users,
                       'purchase_users' : purchase_users})
# Рассчитываем конверсию
result_test['conversion %'] = (result_test['purchase_users'] / result_test['total_users'])

result_test.loc['Разница В-А'] = result_test.loc['B'] - result_test.loc['A']

result_test

,total_users,purchase_users,conversion %
group,,,
A,4952.0,1377.0,0.278069
B,5011.0,1480.0,0.295350
Разница В-А,59.0,103.0,0.017281


**Вывод:** В тестовой группе B наблюдается увеличение пользовательской активности по сравнению с контрольной группой A. Конверсия в покупку выросла с 27,81% до 29,54%, что составляет прирост на 1,73 процентного пункта. Также в группе B больше пользователей совершили покупку - 1480 против 1377 в группе A.

# Оценка результатов A/B-теста

**Гипотеза:** упрощение интерфейса приведёт к тому, что в течение семи дней после регистрации в системе конверсия зарегистрированных пользователей в покупателей увеличится как минимум на три процентных пункта.

* **H₀:** конверсия группы B не выше конверсии группы A;
* **H₁:** конверсия группы B выше конверсии группы A.

In [18]:
# Общее количество пользователей А и В
n_a= result_test.loc['A', 'total_users']
n_b = result_test.loc['B', 'total_users']
# Количество пользователей А и В совершившие покупки
m_a = result_test.loc['A', 'purchase_users']
m_b = result_test.loc['B', 'purchase_users']

alpha = 0.05

stats_ztest, p_value_ztest = proportions_ztest([m_a, m_b],
                                               [n_a , n_b],
                                               alternative = 'smaller')

print(f'p_value = {p_value_ztest}')
print('---'*33)

if p_value_ztest < alpha:
    print(f'Есть основания отвергнуть нулевую гипотезу: '
          f'p-value = {p_value_ztest} < alpha = {alpha}\n'
          'Конверсия в покупку в тестовой группе B статистически значимо '
          'выше, чем в контрольной группе A.')
else:
    print(f'Нет оснований отвергнуть нулевую гипотезу: '
          f'p-value = {p_value_ztest} >= alpha = {alpha}\n'
          'Статистически значимых оснований утверждать, что конверсия '
          'в тестовой группе B выше, чем в контрольной группе A, нет.')
    
print('---'*33)
print('Конверсия в покупку:')
print(round(result_test[['conversion %']]*100, 2))

p_value = 0.028262547212292124
---------------------------------------------------------------------------------------------------
Есть основания отвергнуть нулевую гипотезу: p-value = 0.028262547212292124 < alpha = 0.05
Конверсия в покупку в тестовой группе B статистически значимо выше, чем в контрольной группе A.
---------------------------------------------------------------------------------------------------
Конверсия в покупку:
             conversion %
group                    
A                   27.81
B                   29.54
Разница В-А          1.73


# Выводы по результатам A/B-тестирования

Целью теста была проверка гипотезы о том, что упрощённый интерфейс сайта позволит увеличить конверсию зарегистрированных пользователей в покупателей в течение первых семи дней после регистрации как минимум на 3 процентных пункта.

В ходе проверки корректности теста были исключены 887 пользователей, одновременно участвовавших в конкурирующем тесте. После очистки пользователи были практически равномерно распределены между группами: 49,7% в группе A и 50,3% в группе B. Размер выборки также оказался достаточным для проведения теста: при необходимом размере 3761 пользователя в каждой группе фактически в группе A участвовали 4952 пользователя, а в группе B - 5011 пользователей.

По результатам теста конверсия в покупку в контрольной группе A составила 27,81%, а в тестовой группе B - 29,54%. Таким образом, новый интерфейс увеличил конверсию на 1,73 процентного пункта. Проведённый Z-тест пропорций показал статистически значимое различие между группами (p-value = 0,028 < 0,05).

Таким образом, новый интерфейс действительно оказал положительное влияние на конверсию пользователей в покупку. Однако ожидаемый бизнес-эффект в размере не менее 3 процентных пунктов достигнут не был: фактический прирост составил 1,73 п.п.